# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Masoumeh Davoudi | |
| Member | Armin Maddah Asl | |
| Member *(optional)* | Seyed Mohammad Hossein Ahmadi | |


## 2. Mission Title & Research Question

**Title:** *Minimum-Wage Effects on Labor-Market Outcomes of Young Adults Without a Bachelor's Degree: Evidence from CPS State-Month Panel Data*

**Research question:**  
*To what extent do minimum-wage increases affect employment rates, unemployment rates, and working hours among 16–24 year-olds without a bachelor's degree across U.S. states, and how do demographic and labor-market characteristics jointly explain, predict, and structure state-level labor-market outcomes?*

**Why it matters:**  
*Minimum-wage policy is one of the most direct labor-market interventions available to policymakers, yet its effects on young and less-educated workers remain contested. The CPS state-month panel combines 128 observed minimum-wage increases across 51 jurisdictions over six years (2017–2022) with detailed labor-market and demographic aggregates. By combining causal inference, supervised learning, and clustering, this project estimates the causal effect of minimum-wage changes on youth labor-market outcomes, evaluates how well state-level characteristics predict employment and unemployment rates, and identifies distinct socio-economic profiles of state labor markets over time.*

## 3. Data

**Source(s):**
* This project uses the pre-built CPS Minimum-Wage State-Month Dataset provided with the course materials (`cps_minimum_wage_colleague_package`).
- The processed panel aggregates U.S. Census Bureau Basic Monthly Current Population Survey (CPS) microdata with Vaghul & Zipperer's historical state minimum-wage data (v1.4.0).
- The raw CPS files for 2020–2022 are included in the package; the 2017–2019 data is downloaded from official Census URLs upon reproduction.
- Minimum-wage data source: https://github.com/benzipperer/historicalminwage/releases/tag/v1.4.0
- The final panel is at: `cps_minimum_wage_colleague_package/datasets/processed/cps_state_month_panel.csv`

**Unit of observation:** *One row represents one U.S. state or Washington, DC in one calendar month, summarizing labor-market outcomes for individuals aged 16–24 without a bachelor's degree.*

**Key variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| `year` | Integer | Feature / key | Calendar year |
| `month` | Integer | Feature / key | Calendar month number |
| `state_fips` | Integer | Feature / key | Census state FIPS code |
| `sample_n` | Integer | Feature | Number of eligible CPS records in the state-month |
| `population_weight` | Float | Feature | Sum of CPS person weights |
| `employment_rate` | Float | Target / feature | Weighted share classified as employed |
| `unemployment_rate` | Float | Target / feature | Weighted unemployed share of the target sample |
| `labor_force_rate` | Float | Target / feature | Weighted share employed or unemployed |
| `usual_hours` | Float | Target / feature | Weighted average usual weekly hours |
| `female_share` | Float | Feature | Weighted female share |
| `black_share` | Float | Feature | Weighted Black share |
| `hispanic_share` | Float | Feature | Weighted Hispanic share |
| `mean_age` | Float | Feature | Weighted average age |
| `mean_education` | Float | Feature | Weighted average CPS education code |
| `minimum_wage` | Float | Feature / treatment | Higher of applicable state and federal minimum wage |
| `date` | Date | Key / feature | First day of the state-month observation |
| `log_minimum_wage` | Float | Feature | Natural logarithm of the binding minimum wage |
| `wage_change` | Binary integer | Target / treatment | 1 when the minimum wage increased from the previous month |

**Potential data quality issues:**
* The data is aggregated at the state-month level, not individual-level — within-state variation is lost. All outcomes are weighted means using CPS person weights.

* The target population is restricted to 16–24 year-olds without a bachelor's degree (education code &lt; 43). Findings do not generalize to older or more educated workers.

* The panel covers January 2017 through December 2022, which includes the COVID-19 pandemic period (2020–2021). The pandemic caused large, unusual shifts in employment and labor-force participation that may confound minimum-wage effect estimates.

* `unemployment_rate` is calculated as the weighted unemployed share of the *entire target sample*, not the conventional U-3 rate (unemployed ÷ labor force). This is a deliberate design choice in the source dataset.

* CPS microdata is survey-based and subject to sampling error, especially for smaller states where `sample_n` may be low.

* 2017–2019 CPS data originates from fixed-width `.dat` files, while 2020–2022 data is in CSV format. The reproduction script handles both, but subtle differences in variable coding across formats could introduce inconsistencies.

* Minimum wage is the *binding* rate (max of state and federal). Some jurisdictions had preempted local minimum wages during this period; the dataset does not capture city- or county-level minimum wages that exceed the state rate.

In [1]:
# Data loading & first inspection — CPS Minimum-Wage State-Month Panel
import pandas as pd
import numpy as np

# Path to the processed panel (adjust if your working directory differs)
PANEL_PATH = "../../cps_minimum_wage_colleague_package/datasets/processed/cps_state_month_panel.csv"

df = pd.read_csv(PANEL_PATH, parse_dates=["date"])

print("Shape of dataset:", df.shape)
print("Period:", df["date"].min(), "to", df["date"].max())
print("Geographic units:", df["state_fips"].nunique())
print("Months per unit (balanced?):", df.groupby("state_fips").size().unique())

display(df.head())

print("\nDataset information:")
print(df.info())

print("\nSummary statistics:")
display(df.describe())

print("\nMissing values:")
missing_table = pd.DataFrame(
    {"missing_count": df.isna().sum(), "missing_share": df.isna().mean()}
).sort_values("missing_share", ascending=False)
display(missing_table)

print("\nNumber of observed minimum-wage increases:", df["wage_change"].sum())
print("\nUnique minimum wage values:", sorted(df["minimum_wage"].unique()))

Shape of dataset: (3672, 18)
Period: 2017-01-01 00:00:00 to 2022-12-01 00:00:00
Geographic units: 51
Months per unit (balanced?): [72]


,year,month,state_fips,sample_n,population_weight,employment_rate,unemployment_rate,labor_force_rate,usual_hours,female_share,black_share,hispanic_share,mean_age,mean_education,minimum_wage,date,log_minimum_wage,wage_change
0,2017,1,1,256,5.391558e+09,0.346764,0.064394,0.411157,35.394742,0.562053,0.322628,0.057522,19.961078,38.318510,7.25,2017-01-01,1.981001,0
1,2017,2,1,265,5.604985e+09,0.428163,0.030057,0.458220,33.618857,0.519350,0.321099,0.027033,19.924063,38.400138,7.25,2017-02-01,1.981001,0
2,2017,3,1,266,5.620319e+09,0.434792,0.050956,0.485747,33.701711,0.498903,0.321184,0.019983,20.063628,38.404987,7.25,2017-03-01,1.981001,0
3,2017,4,1,271,5.256655e+09,0.430605,0.060219,0.490824,35.200129,0.499722,0.326115,0.031351,20.010448,38.255047,7.25,2017-04-01,1.981001,0
4,2017,5,1,242,5.059472e+09,0.453535,0.038751,0.492286,34.359847,0.481167,0.361212,0.037955,20.138618,38.383280,7.25,2017-05-01,1.981001,0



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3672 entries, 0 to 3671
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   year               3672 non-null   int64         
 1   month              3672 non-null   int64         
 2   state_fips         3672 non-null   int64         
 3   sample_n           3672 non-null   int64         
 4   population_weight  3672 non-null   float64       
 5   employment_rate    3672 non-null   float64       
 6   unemployment_rate  3672 non-null   float64       
 7   labor_force_rate   3672 non-null   float64       
 8   usual_hours        3672 non-null   float64       
 9   female_share       3672 non-null   float64       
 10  black_share        3672 non-null   float64       
 11  hispanic_share     3672 non-null   float64       
 12  mean_age           3672 non-null   float64       
 13  mean_education     3672 non-null   float6

,year,month,state_fips,sample_n,population_weight,employment_rate,unemployment_rate,labor_force_rate,usual_hours,female_share,black_share,hispanic_share,mean_age,mean_education,minimum_wage,date,log_minimum_wage,wage_change
count,3672.000000,3672.000000,3672.000000,3672.000000,3.672000e+03,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672.000000,3672,3672.000000,3672.000000
mean,2019.500000,6.500000,28.960784,209.424292,6.662161e+09,0.498188,0.055433,0.553621,31.484673,0.487255,0.140756,0.165011,19.649351,38.442742,9.086569,2019-12-16 11:20:00,2.184028,0.034858
min,2017.000000,1.000000,1.000000,36.000000,3.377672e+08,0.143799,0.000000,0.253853,20.895205,0.343554,0.000000,0.000000,18.503327,37.733334,7.250000,2017-01-01 00:00:00,1.981001,0.000000
25%,2018.000000,3.750000,16.000000,133.000000,1.821215e+09,0.437107,0.036615,0.494133,29.840900,0.465673,0.040725,0.070804,19.495499,38.311147,7.250000,2018-06-23 12:00:00,1.981001,0.000000
50%,2019.500000,6.500000,29.000000,172.000000,4.681983e+09,0.496366,0.050609,0.551093,31.669969,0.487090,0.099455,0.129921,19.655605,38.448189,8.600000,2019-12-16 12:00:00,2.151762,0.000000
75%,2021.000000,9.250000,42.000000,232.000000,7.976073e+09,0.560118,0.066755,0.612268,33.300085,0.507882,0.199235,0.216760,19.809658,38.578871,10.340000,2021-06-08 12:00:00,2.336020,0.000000
max,2022.000000,12.000000,56.000000,1213.000000,4.587020e+10,0.788129,0.273366,0.813769,40.558869,0.693624,0.773469,0.663432,20.612396,39.203099,16.100000,2022-12-01 00:00:00,2.778819,1.000000
std,1.708058,3.452523,15.678970,151.706724,7.704083e+09,0.092054,0.028741,0.086003,2.610597,0.033913,0.133366,0.130614,0.246129,0.198120,2.022925,NaN,0.209269,0.183446



Missing values:


,missing_count,missing_share
year,0,0.0
month,0,0.0
state_fips,0,0.0
sample_n,0,0.0
population_weight,0,0.0
employment_rate,0,0.0
unemployment_rate,0,0.0
labor_force_rate,0,0.0
usual_hours,0,0.0
female_share,0,0.0



Number of observed minimum-wage increases: 128

Unique minimum wage values: [np.float64(7.25), np.float64(7.5), np.float64(7.699999809265136), np.float64(7.849999904632568), np.float64(8.100000381469727), np.float64(8.149999618530273), np.float64(8.25), np.float64(8.300000190734863), np.float64(8.439999580383299), np.float64(8.460000038146973), np.float64(8.5), np.float64(8.550000190734863), np.float64(8.5600004196167), np.float64(8.600000381469727), np.float64(8.649999618530273), np.float64(8.699999809265137), np.float64(8.75), np.float64(8.800000190734863), np.float64(8.850000381469727), np.float64(8.899999618530273), np.float64(9.0), np.float64(9.100000381469728), np.float64(9.199999809265137), np.float64(9.25), np.float64(9.300000190734863), np.float64(9.449999809265137), np.float64(9.5), np.float64(9.600000381469728), np.float64(9.649999618530272), np.float64(9.699999809265137), np.float64(9.75), np.float64(9.800000190734863), np.float64(9.840000152587889), np.float64(9.859999656

## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference

- [+] Causal graph / DAG (DoWhy)
- [+] Backdoor adjustment
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [+] Other: stacked cohort difference-in-differences, event study, and algorithmic refutation tests

*Justification:*  
We estimate the causal effect of state minimum-wage increases on employment, labor-force participation, and usual weekly hours among 16–24 year-olds without a bachelor's degree. A DAG makes the identifying assumptions explicit. The primary design is a stacked cohort difference-in-differences model that compares states experiencing their first observed minimum-wage increase with never-treated states within a six-month event window. Stack-specific state and calendar-month fixed effects block observed and time-invariant unobserved confounders, while time-varying population composition controls (mean age, education, female/Black/Hispanic shares) address shifting demographics. Supporting continuous-treatment two-way fixed-effects regressions with state and month fixed effects and CPS person weights serve as robustness checks. Causal claims are validated using DoWhy placebo-treatment, random-common-cause, and data-subset refutation tests, along with leave-one-state-out and event-study pre-trend tests.

Instrumental variables and propensity-score stratification are not selected because no defensible instrument is available and propensity scores are not the natural primary method for a staggered state-level policy treatment.

### 4b. Supervised Learning

- [ ] Linear / Ridge / Lasso regression
- [ ] Logistic regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [+] Decision Tree / Random Forest
- [ ] Neural network (regression or classification)
- [+] Other: Gradient Boosting Regression

*Justification:*  
We use Random Forest Regression and Gradient Boosting Regression to predict continuous labor-market outcomes such as `employment_rate`, `unemployment_rate`, `labor_force_rate`, and `usual_hours`. These tree-based ensemble methods are suitable because they can capture non-linear relationships and interactions between minimum wages, demographic composition, state characteristics, and time trends without requiring a strict linear model.

### 4c. Unsupervised Learning / Generative Models

- [+] K-Means clustering
- [+] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [+] Other: PCA for dimensionality reduction and cluster visualization

*Justification:*  
We will use unsupervised learning to identify recurring state-month labor-market regimes across U.S. states from 2017 to 2022. The clustering will use only standardized numeric variables, including employment rate, unemployment rate, labor-force participation, usual weekly hours, minimum wage level, and demographic composition. We will exclude identifiers such as `state_fips`, `date`, `year`, and `month` from the clustering features and use them only for post-cluster interpretation. PCA will be used for visualization, K-Means will be the main clustering method, and hierarchical clustering will be used as a robustness check. The resulting clusters will be interpreted as labor-market regimes rather than causal effects.

## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- **The metric(s) you will use for each model:**  
  - For Random Forest Regression and Gradient Boosting Regression, we will use out-of-sample RMSE, MAE, and R² to evaluate predictions of `employment_rate`, `unemployment_rate`, `labor_force_rate`, and `usual_hours`.
  - For unsupervised learning, we will evaluate clustering quality using the elbow method, silhouette score, PCA visualization, and interpretability of the resulting clusters. The final number of clusters will be selected based on both quantitative scores and whether the clusters correspond to meaningful labor-market regimes.   
  - For the causal inference component, we will report the estimated effect size (in percentage-point change), standard errors, confidence intervals, and p-values for the minimum-wage treatment. We will also report joint pre-trend test p-values from the event study and the results of DoWhy algorithmic refutation tests (placebo treatment, random common cause, data-subset).

- **How you will validate causal claims:**    
  - We will validate the causal analysis by using a DAG to identify plausible confounders and applying a stacked cohort difference-in-differences design with state and month fixed effects. The causal claims will be validated using at least two algorithmic refutation tests via DoWhy — specifically a placebo treatment test (assigning the treatment randomly to verify the estimator returns zero) and a random common cause test (adding an unobserved confounder to check estimate stability). Additionally, we will run an event study with a joint pre-trend test to evaluate whether treatment and control states followed parallel trends before minimum-wage increases, and a leave-one-state-out sensitivity analysis.

- **Any baselines or benchmarks you will compare against:**  
  - We will compare Random Forest Regression and Gradient Boosting Regression against a simple mean-prediction baseline.
  - For unsupervised learning, we will compare several values of `k` in K-Means and check whether similar cluster patterns appear when using hierarchical clustering as a robustness check. 
  - For causal inference, we will compare estimates from the stacked cohort difference-in-differences design against a continuous-treatment two-way fixed-effects specification, as well as an unadjusted (no controls) baseline. We will also compare across robustness checks such as excluding COVID-period months and including state-specific linear trends.

## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Armin & Masoumeh| Data collection & cleaning |
| 2 | Masoumeh  | EDA |
| 3 | Armin | Causal inference block |
| 4 | Seyed Mohammad Hossein | Supervised learning block |
| 5 | Masoumeh | Unsupervised / generative block |
| 6 | All team members  | Synthesis & write-up |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
